In [7]:
from pathlib import Path
import pandas as pd

BASE = Path.cwd().parent
RAW  = BASE / "data" / "raw"


datasets = {
    "fund_master":         "01_fund_master.csv",
    "nav_history":         "02_nav_history.csv",
    "aum_by_fund_house":   "03_aum_by_fund_house.csv",
    "monthly_sip_inflows": "04_monthly_sip_inflows.csv",
    "category_inflows":    "05_category_inflows.csv",
    "industry_folio_count":"06_industry_folio_count.csv",
    "scheme_performance":  "07_scheme_performance.csv",
    "investor_transactions":"08_investor_transactions.csv",
    "portfolio_holdings":  "09_portfolio_holdings.csv",
    "benchmark_indices":   "10_benchmark_indices.csv",
}

dfs = {}
for name, fname in datasets.items():
    df = pd.read_csv(RAW / fname)
    dfs[name] = df
    print(f"\n{'='*40}")
    print(f"{name}: {df.shape}")
    print(df.dtypes)
    print(df.head(2))




fund_master: (40, 15)
amfi_code               int64
fund_house                str
scheme_name               str
category                  str
sub_category              str
plan                      str
launch_date               str
benchmark                 str
expense_ratio_pct     float64
exit_load_pct         float64
min_sip_amount          int64
min_lumpsum_amount      int64
fund_manager              str
risk_category             str
sebi_category_code        str
dtype: object
   amfi_code       fund_house                                scheme_name  \
0     119551  SBI Mutual Fund  SBI Bluechip Fund - Regular Plan - Growth   
1     119552  SBI Mutual Fund   SBI Bluechip Fund - Direct Plan - Growth   

  category sub_category     plan launch_date      benchmark  \
0   Equity    Large Cap  Regular  2006-02-14  NIFTY 100 TRI   
1   Equity    Large Cap   Direct  2013-01-01  NIFTY 100 TRI   

   expense_ratio_pct  exit_load_pct  min_sip_amount  min_lumpsum_amount  \
0               1.5

In [10]:
import requests, time
import pandas as pd
from pathlib import Path

RAW = Path.cwd().parent/ "data" / "raw"

SCHEMES = {
    119551: "SBI_Bluechip",
    120503: "ICICI_Bluechip",
    118632: "Nippon_LargeCap",
    119092: "Axis_Bluechip",
    120841: "Kotak_Bluechip",
    125497: "HDFC_Top100",     # anchor scheme from project spec
}

def fetch_nav(scheme_code: int, retries: int = 3) -> pd.DataFrame:
    url = f"https://api.mfapi.in/mf/{scheme_code}"
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=15)
            r.raise_for_status()
            data = r.json()["data"]           # list of {date, nav}
            df = pd.DataFrame(data)
            df["date"] = pd.to_datetime(df["date"], dayfirst=True)
            df["nav"]  = df["nav"].astype(float)
            df["amfi_code"] = scheme_code
            return df.sort_values("date").reset_index(drop=True)
        except Exception as e:
            print(f"Attempt {attempt+1} failed for {scheme_code}: {e}")
            time.sleep(2)
    return pd.DataFrame()

if __name__ == "__main__":
    for code, name in SCHEMES.items():
        df = fetch_nav(code)
        if not df.empty:
            out = RAW / f"live_nav_{name}_{code}.csv"
            df.to_csv(out, index=False)
            print(f"Saved {len(df)} rows → {out.name}")

Saved 3236 rows → live_nav_SBI_Bluechip_119551.csv
Saved 3307 rows → live_nav_ICICI_Bluechip_120503.csv
Saved 3298 rows → live_nav_Nippon_LargeCap_118632.csv
Saved 3565 rows → live_nav_Axis_Bluechip_119092.csv
Saved 3301 rows → live_nav_Kotak_Bluechip_120841.csv
Saved 3091 rows → live_nav_HDFC_Top100_125497.csv


In [11]:
master_codes = set(dfs["fund_master"]["amfi_code"])
nav_codes    = set(dfs["nav_history"]["amfi_code"])
missing = master_codes - nav_codes
print(f"Codes in master but missing from NAV: {missing or 'None — all good!'}")

print(dfs["fund_master"]["fund_house"].value_counts())
print(dfs["fund_master"]["category"].value_counts())
print(dfs["fund_master"]["risk_category"].value_counts())

Codes in master but missing from NAV: None — all good!
fund_house
SBI Mutual Fund             5
HDFC Mutual Fund            5
ICICI Prudential MF         5
Nippon India MF             5
Kotak Mahindra MF           4
Axis Mutual Fund            4
Aditya Birla Sun Life MF    3
UTI Mutual Fund             3
Mirae Asset MF              3
DSP Mutual Fund             3
Name: count, dtype: int64
category
Equity    34
Debt       6
Name: count, dtype: int64
risk_category
Moderate           16
High                8
Very High           6
Low                 6
Moderately High     4
Name: count, dtype: int64
